# 03 Feature Engineering — Scope and Leakage Rules

This notebook creates deterministic, leakage-free feature matrices for the MoA prediction project.

The goal is to build feature-set versions that can later be compared during model training:

- FE_V0: raw metadata + gene + cell features
- FE_V1: FE_V0 + row-wise gene/cell summary features
- FE_V2: FE_V1 + small gene-cell interaction features
- FE_V3: optional metadata interaction features

This notebook will not train models, fit scalers, fit PCA, fit quantile transformers, fit feature selectors, or use any target-derived information as input features.

All learned transformations such as scaling, PCA, quantile transformation, variance filtering, and supervised feature selection must be fitted only inside the validation/modeling pipeline.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project root detection
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EXPERIMENT_LOG_DIR = PROJECT_ROOT / "outputs" / "experiment_logs"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_LOG_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_INTERIM_DIR:", DATA_INTERIM_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_LOG_DIR:", EXPERIMENT_LOG_DIR)
print("FIGURE_DIR:", FIGURE_DIR)

PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response
DATA_RAW_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\raw
DATA_INTERIM_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim
DATA_PROCESSED_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\processed
EXPERIMENT_LOG_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\outputs\experiment_logs
FIGURE_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\reports\figures


In [3]:
feature_engineering_scope_table = pd.DataFrame({
    "area": [
        "Notebook purpose",
        "Allowed feature type",
        "Baseline feature versions",
        "Model training",
        "Learned transformations",
        "Target-derived features",
        "Nonscored targets",
        "Drug ID",
        "Control-row prediction rule"
    ],
    "rule": [
        "Create leakage-free deterministic feature matrices",
        "Only input-derived, row-wise features",
        "FE_V0, FE_V1, FE_V2, optional FE_V3",
        "Not performed in this notebook",
        "Not fitted in this notebook",
        "Never used as model input",
        "Never used as baseline input features",
        "Not used as baseline model input",
        "Documented here, applied only after prediction"
    ],
    "reason": [
        "Feature engineering should stay separate from modeling",
        "Safe because each feature uses only the same sample's input values",
        "Allows clean model comparison later",
        "Model training belongs to model training notebook",
        "Scaler/PCA/Quantile/selection must be fitted inside CV pipeline",
        "Would leak label information",
        "Nonscored labels are auxiliary targets only, not inputs",
        "Drug ID is useful for validation diagnostics but risky as baseline input",
        "ctl_vehicle zeroing is an inference/submission post-processing rule"
    ]
})

feature_engineering_scope_table

,area,rule,reason
0,Notebook purpose,Create leakage-free deterministic feature matr...,Feature engineering should stay separate from ...
1,Allowed feature type,"Only input-derived, row-wise features",Safe because each feature uses only the same s...
2,Baseline feature versions,"FE_V0, FE_V1, FE_V2, optional FE_V3",Allows clean model comparison later
3,Model training,Not performed in this notebook,Model training belongs to model training notebook
4,Learned transformations,Not fitted in this notebook,Scaler/PCA/Quantile/selection must be fitted i...
5,Target-derived features,Never used as model input,Would leak label information
6,Nonscored targets,Never used as baseline input features,"Nonscored labels are auxiliary targets only, n..."
7,Drug ID,Not used as baseline model input,Drug ID is useful for validation diagnostics b...
8,Control-row prediction rule,"Documented here, applied only after prediction",ctl_vehicle zeroing is an inference/submission...


In [4]:
hard_leakage_rules = pd.DataFrame({
    "rule_id": [
        "L001",
        "L002",
        "L003",
        "L004",
        "L005",
        "L006",
        "L007",
        "L008",
        "L009",
        "L010"
    ],
    "leakage_rule": [
        "No scored target column can enter X",
        "No nonscored target column can enter X",
        "No target-derived count or flag can enter X",
        "No fold-level target statistic can enter X",
        "No target co-occurrence feature can enter X",
        "No scaler is fitted before cross-validation",
        "No PCA is fitted before cross-validation",
        "No QuantileTransformer is fitted before cross-validation",
        "No feature selector is fitted before cross-validation",
        "No drug_id is used as a baseline input feature"
    ],
    "forbidden_examples": [
        "nfkb_inhibitor, proteasome_inhibitor, etc.",
        "any nonscored target label",
        "active_target_count, zero_label_flag, multi_label_count",
        "validation_positive_count, rare_label_fold_count",
        "target_pair_count, label_cluster_id",
        "StandardScaler.fit(full_train)",
        "PCA.fit(full_train)",
        "QuantileTransformer.fit(full_train)",
        "SelectKBest.fit(full_train, y)",
        "drug_id one-hot as baseline X"
    ],
    "safe_alternative": [
        "Use targets only as y",
        "Use nonscored only as advanced auxiliary y",
        "Use gene/cell input-derived summaries",
        "Use fold statistics only for diagnostics",
        "Use co-occurrence only for EDA interpretation",
        "Fit scaler inside modeling pipeline",
        "Fit PCA inside modeling pipeline",
        "Fit QuantileTransformer inside modeling pipeline",
        "Fit selector inside modeling pipeline",
        "Use drug_id only for validation-risk diagnostics"
    ]
})

hard_leakage_rules

,rule_id,leakage_rule,forbidden_examples,safe_alternative
0,L001,No scored target column can enter X,"nfkb_inhibitor, proteasome_inhibitor, etc.",Use targets only as y
1,L002,No nonscored target column can enter X,any nonscored target label,Use nonscored only as advanced auxiliary y
2,L003,No target-derived count or flag can enter X,"active_target_count, zero_label_flag, multi_la...",Use gene/cell input-derived summaries
3,L004,No fold-level target statistic can enter X,"validation_positive_count, rare_label_fold_count",Use fold statistics only for diagnostics
4,L005,No target co-occurrence feature can enter X,"target_pair_count, label_cluster_id",Use co-occurrence only for EDA interpretation
5,L006,No scaler is fitted before cross-validation,StandardScaler.fit(full_train),Fit scaler inside modeling pipeline
6,L007,No PCA is fitted before cross-validation,PCA.fit(full_train),Fit PCA inside modeling pipeline
7,L008,No QuantileTransformer is fitted before cross-...,QuantileTransformer.fit(full_train),Fit QuantileTransformer inside modeling pipeline
8,L009,No feature selector is fitted before cross-val...,"SelectKBest.fit(full_train, y)",Fit selector inside modeling pipeline
9,L010,No drug_id is used as a baseline input feature,drug_id one-hot as baseline X,Use drug_id only for validation-risk diagnostics


In [5]:
feature_category_rules = pd.DataFrame({
    "feature_category": [
        "Raw metadata",
        "Raw gene features",
        "Raw cell features",
        "Gene summary features",
        "Cell summary features",
        "Gene-cell interaction features",
        "Metadata interaction features",
        "Scaling",
        "PCA",
        "Quantile transformation",
        "Feature selection",
        "Nonscored targets",
        "Scored targets",
        "Drug ID",
        "Control-row zeroing"
    ],
    "status": [
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "optional",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "forbidden-as-input",
        "forbidden-as-input",
        "diagnostic-only",
        "inference-only"
    ],
    "where_used": [
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V1 onward",
        "FE_V1 onward",
        "FE_V2 onward",
        "optional FE_V3",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "advanced auxiliary output only",
        "y target only",
        "validation-risk analysis only",
        "submission post-processing"
    ],
    "notes": [
        "cp_type, cp_time, cp_dose",
        "g-* columns",
        "c-* columns",
        "row-wise deterministic",
        "row-wise deterministic",
        "small audited set only",
        "keep small; avoid sparse explosion",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "never concatenate into X",
        "never concatenate into X",
        "not a baseline feature",
        "apply after prediction, not during feature creation"
    ]
})

feature_category_rules

,feature_category,status,where_used,notes
0,Raw metadata,allowed,FE_V0 onward,"cp_type, cp_time, cp_dose"
1,Raw gene features,allowed,FE_V0 onward,g-* columns
2,Raw cell features,allowed,FE_V0 onward,c-* columns
3,Gene summary features,allowed,FE_V1 onward,row-wise deterministic
4,Cell summary features,allowed,FE_V1 onward,row-wise deterministic
5,Gene-cell interaction features,allowed,FE_V2 onward,small audited set only
6,Metadata interaction features,optional,optional FE_V3,keep small; avoid sparse explosion
7,Scaling,pipeline-only,modeling pipeline,fit on training folds only
8,PCA,pipeline-only,modeling pipeline,fit on training folds only
9,Quantile transformation,pipeline-only,modeling pipeline,fit on training folds only


In [6]:
feature_engineering_scope_table.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_engineering_scope_table.csv",
    index=False
)

hard_leakage_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_hard_leakage_rules.csv",
    index=False
)

feature_category_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_category_rules.csv",
    index=False
)

print("Step 1 scope and leakage-rule files saved successfully.")

Step 1 scope and leakage-rule files saved successfully.


In [7]:
step_1_status = {
    "step": "Step 1 — Notebook Scope and Leakage Rules",
    "status": "completed",
    "model_training_started": False,
    "learned_transform_fitted": False,
    "target_derived_features_allowed": False,
    "nonscored_targets_allowed_as_input": False,
    "drug_id_allowed_as_baseline_input": False,
    "ready_for_step_2": True,
    "next_step": "Step 2 — Load Clean Interim Data"
}

step_1_status

{'step': 'Step 1 — Notebook Scope and Leakage Rules',
 'status': 'completed',
 'model_training_started': False,
 'learned_transform_fitted': False,
 'target_derived_features_allowed': False,
 'nonscored_targets_allowed_as_input': False,
 'drug_id_allowed_as_baseline_input': False,
 'ready_for_step_2': True,
 'next_step': 'Step 2 — Load Clean Interim Data'}

## Step 2 — Load Clean Interim Data

This step loads the clean interim datasets created during data integration.

This notebook does not repeat raw data integration.  
The goal is to start feature engineering from already validated clean assets.

Loaded assets:

- clean train features
- clean test features
- scored targets
- nonscored targets
- train drug table
- feature group dictionary
- sample submission

In [8]:
expected_interim_files = {
    "train_features_clean": DATA_INTERIM_DIR / "train_features_clean.parquet",
    "test_features_clean": DATA_INTERIM_DIR / "test_features_clean.parquet",
    "y_scored": DATA_INTERIM_DIR / "y_scored.parquet",
    "y_nonscored": DATA_INTERIM_DIR / "y_nonscored.parquet",
    "train_drug_clean": DATA_INTERIM_DIR / "train_drug_clean.parquet",
    "feature_groups": DATA_INTERIM_DIR / "feature_groups.json"
}

expected_raw_files = {
    "sample_submission": DATA_RAW_DIR / "sample_submission.csv"
}

file_availability_audit = []

for file_name, file_path in {**expected_interim_files, **expected_raw_files}.items():
    file_availability_audit.append({
        "file_name": file_name,
        "expected_path": str(file_path),
        "exists": file_path.exists()
    })

file_availability_audit = pd.DataFrame(file_availability_audit)

file_availability_audit

,file_name,expected_path,exists
0,train_features_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
1,test_features_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
2,y_scored,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
3,y_nonscored,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
4,train_drug_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
5,feature_groups,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
6,sample_submission,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True


In [9]:
missing_files = file_availability_audit.loc[
    file_availability_audit["exists"] == False,
    ["file_name", "expected_path"]
]

if len(missing_files) > 0:
    display(missing_files)
    raise FileNotFoundError("Some required clean/interim files are missing. Check paths before continuing.")

print("All required clean/interim files are available.")

All required clean/interim files are available.


In [11]:
def read_table_with_repair(parquet_path, raw_csv_path=None, dataset_name="dataset"):
    """
    Try to read parquet safely.
    If parquet fails, optionally reload from raw CSV and rewrite parquet.
    """
    parquet_path = Path(parquet_path)
    raw_csv_path = Path(raw_csv_path) if raw_csv_path is not None else None
    
    try:
        df = pd.read_parquet(parquet_path, engine="pyarrow")
        print(f"{dataset_name}: loaded from parquet with pyarrow")
        return df
    
    except Exception as pyarrow_error:
        print(f"{dataset_name}: pyarrow parquet read failed.")
        print("Error:", pyarrow_error)
        
        try:
            df = pd.read_parquet(parquet_path, engine="fastparquet")
            print(f"{dataset_name}: loaded from parquet with fastparquet")
            return df
        
        except Exception as fastparquet_error:
            print(f"{dataset_name}: fastparquet parquet read also failed.")
            print("Error:", fastparquet_error)
            
            if raw_csv_path is not None and raw_csv_path.exists():
                print(f"{dataset_name}: loading from raw CSV and repairing parquet...")
                df = pd.read_csv(raw_csv_path)
                
                df.to_parquet(
                    parquet_path,
                    index=False,
                    engine="pyarrow"
                )
                
                print(f"{dataset_name}: repaired parquet saved to {parquet_path}")
                return df
            
            raise RuntimeError(
                f"{dataset_name}: Could not read parquet and no valid raw CSV fallback found."
            )


raw_csv_fallback_files = {
    "train_features_clean": DATA_RAW_DIR / "train_features.csv",
    "test_features_clean": DATA_RAW_DIR / "test_features.csv",
    "y_scored": DATA_RAW_DIR / "train_targets_scored.csv",
    "y_nonscored": DATA_RAW_DIR / "train_targets_nonscored.csv",
    "train_drug_clean": DATA_RAW_DIR / "train_drug.csv"
}


train_features_clean = read_table_with_repair(
    expected_interim_files["train_features_clean"],
    raw_csv_fallback_files["train_features_clean"],
    "train_features_clean"
)

test_features_clean = read_table_with_repair(
    expected_interim_files["test_features_clean"],
    raw_csv_fallback_files["test_features_clean"],
    "test_features_clean"
)

y_scored = read_table_with_repair(
    expected_interim_files["y_scored"],
    raw_csv_fallback_files["y_scored"],
    "y_scored"
)

y_nonscored = read_table_with_repair(
    expected_interim_files["y_nonscored"],
    raw_csv_fallback_files["y_nonscored"],
    "y_nonscored"
)

train_drug_clean = read_table_with_repair(
    expected_interim_files["train_drug_clean"],
    raw_csv_fallback_files["train_drug_clean"],
    "train_drug_clean"
)

sample_submission = pd.read_csv(expected_raw_files["sample_submission"])

with open(expected_interim_files["feature_groups"], "r", encoding="utf-8") as f:
    feature_groups = json.load(f)

print("Clean data loaded successfully.")

train_features_clean: pyarrow parquet read failed.
Error: Repetition level histogram size mismatch
train_features_clean: fastparquet parquet read also failed.
Error: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
train_features_clean: loading from raw CSV and repairing parquet...
train_features_clean: repaired parquet saved to C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim\train_features_clean.parquet
test_features_clean: pyarrow parquet read failed.
Error: Repetition level histogram size mismatch
test_features_clean: fastparquet parquet read also failed.
Error: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
test_features_clean: loading from raw CSV and repairing parquet...
test_features_clean: repaired parquet saved to C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Mach

In [13]:
data_shape_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean",
        "sample_submission"
    ],
    "rows": [
        train_features_clean.shape[0],
        test_features_clean.shape[0],
        y_scored.shape[0],
        y_nonscored.shape[0],
        train_drug_clean.shape[0],
        sample_submission.shape[0]
    ],
    "columns": [
        train_features_clean.shape[1],
        test_features_clean.shape[1],
        y_scored.shape[1],
        y_nonscored.shape[1],
        train_drug_clean.shape[1],
        sample_submission.shape[1]
    ]
})

data_shape_audit

,dataset,rows,columns
0,train_features_clean,23814,876
1,test_features_clean,3982,876
2,y_scored,23814,207
3,y_nonscored,23814,403
4,train_drug_clean,23814,2
5,sample_submission,3982,207


In [14]:
def get_feature_group(feature_groups_dict, possible_keys, default=None):
    for key in possible_keys:
        if key in feature_groups_dict:
            return feature_groups_dict[key]
    return default if default is not None else []


ID_COL = "sig_id"

metadata_features = get_feature_group(
    feature_groups,
    ["metadata_features", "metadata", "meta_features"],
    default=["cp_type", "cp_time", "cp_dose"]
)

gene_features = get_feature_group(
    feature_groups,
    ["gene_features", "gene", "genes"],
    default=[col for col in train_features_clean.columns if col.startswith("g-")]
)

cell_features = get_feature_group(
    feature_groups,
    ["cell_features", "cell", "cells"],
    default=[col for col in train_features_clean.columns if col.startswith("c-")]
)

scored_target_features = get_feature_group(
    feature_groups,
    ["scored_target_features", "scored_targets", "target_features", "targets_scored"],
    default=[col for col in y_scored.columns if col != ID_COL]
)

nonscored_target_features = get_feature_group(
    feature_groups,
    ["nonscored_target_features", "nonscored_targets", "target_features_nonscored", "targets_nonscored"],
    default=[col for col in y_nonscored.columns if col != ID_COL]
)

feature_group_count_audit = pd.DataFrame({
    "feature_group": [
        "metadata_features",
        "gene_features",
        "cell_features",
        "scored_target_features",
        "nonscored_target_features"
    ],
    "count": [
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features)
    ]
})

feature_group_count_audit

,feature_group,count
0,metadata_features,3
1,gene_features,772
2,cell_features,100
3,scored_target_features,206
4,nonscored_target_features,402


In [15]:
id_alignment_audit = {
    "train_sig_id_unique": train_features_clean[ID_COL].is_unique,
    "test_sig_id_unique": test_features_clean[ID_COL].is_unique,
    "y_scored_sig_id_unique": y_scored[ID_COL].is_unique,
    "y_nonscored_sig_id_unique": y_nonscored[ID_COL].is_unique,
    "train_drug_sig_id_unique": train_drug_clean[ID_COL].is_unique,
    "sample_submission_sig_id_unique": sample_submission[ID_COL].is_unique,
    
    "train_vs_y_scored_order_match": train_features_clean[ID_COL].tolist() == y_scored[ID_COL].tolist(),
    "train_vs_y_nonscored_order_match": train_features_clean[ID_COL].tolist() == y_nonscored[ID_COL].tolist(),
    "train_vs_drug_order_match": train_features_clean[ID_COL].tolist() == train_drug_clean[ID_COL].tolist(),
    "test_vs_submission_order_match": test_features_clean[ID_COL].tolist() == sample_submission[ID_COL].tolist(),
    
    "train_rows_match_y_scored": train_features_clean.shape[0] == y_scored.shape[0],
    "train_rows_match_y_nonscored": train_features_clean.shape[0] == y_nonscored.shape[0],
    "train_rows_match_train_drug": train_features_clean.shape[0] == train_drug_clean.shape[0],
    "test_rows_match_submission": test_features_clean.shape[0] == sample_submission.shape[0]
}

id_alignment_audit

{'train_sig_id_unique': True,
 'test_sig_id_unique': True,
 'y_scored_sig_id_unique': True,
 'y_nonscored_sig_id_unique': True,
 'train_drug_sig_id_unique': True,
 'sample_submission_sig_id_unique': True,
 'train_vs_y_scored_order_match': True,
 'train_vs_y_nonscored_order_match': True,
 'train_vs_drug_order_match': True,
 'test_vs_submission_order_match': True,
 'train_rows_match_y_scored': True,
 'train_rows_match_y_nonscored': True,
 'train_rows_match_train_drug': True,
 'test_rows_match_submission': True}

In [16]:
train_feature_columns_without_id = [col for col in train_features_clean.columns if col != ID_COL]
test_feature_columns_without_id = [col for col in test_features_clean.columns if col != ID_COL]

feature_column_alignment_audit = {
    "train_feature_count_without_id": len(train_feature_columns_without_id),
    "test_feature_count_without_id": len(test_feature_columns_without_id),
    "train_test_feature_columns_match": train_feature_columns_without_id == test_feature_columns_without_id,
    "metadata_columns_present": all(col in train_features_clean.columns for col in metadata_features),
    "gene_columns_present": all(col in train_features_clean.columns for col in gene_features),
    "cell_columns_present": all(col in train_features_clean.columns for col in cell_features),
    "sample_submission_target_count": sample_submission.shape[1] - 1,
    "scored_target_count": len(scored_target_features),
    "submission_target_columns_match_scored_targets": sample_submission.columns[1:].tolist() == scored_target_features
}

feature_column_alignment_audit

{'train_feature_count_without_id': 875,
 'test_feature_count_without_id': 875,
 'train_test_feature_columns_match': True,
 'metadata_columns_present': True,
 'gene_columns_present': True,
 'cell_columns_present': True,
 'sample_submission_target_count': 206,
 'scored_target_count': 206,
 'submission_target_columns_match_scored_targets': True}

In [17]:
missing_value_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean",
        "sample_submission"
    ],
    "missing_values": [
        int(train_features_clean.isna().sum().sum()),
        int(test_features_clean.isna().sum().sum()),
        int(y_scored.isna().sum().sum()),
        int(y_nonscored.isna().sum().sum()),
        int(train_drug_clean.isna().sum().sum()),
        int(sample_submission.isna().sum().sum())
    ]
})

missing_value_audit

,dataset,missing_values
0,train_features_clean,0
1,test_features_clean,0
2,y_scored,0
3,y_nonscored,0
4,train_drug_clean,0
5,sample_submission,0


In [18]:
assert id_alignment_audit["train_sig_id_unique"], "train_features_clean sig_id is not unique"
assert id_alignment_audit["test_sig_id_unique"], "test_features_clean sig_id is not unique"
assert id_alignment_audit["train_vs_y_scored_order_match"], "train and y_scored sig_id order mismatch"
assert id_alignment_audit["train_vs_y_nonscored_order_match"], "train and y_nonscored sig_id order mismatch"
assert id_alignment_audit["train_vs_drug_order_match"], "train and train_drug sig_id order mismatch"
assert id_alignment_audit["test_vs_submission_order_match"], "test and sample_submission sig_id order mismatch"

assert feature_column_alignment_audit["train_test_feature_columns_match"], "train/test feature columns do not match"
assert feature_column_alignment_audit["metadata_columns_present"], "metadata columns missing"
assert feature_column_alignment_audit["gene_columns_present"], "gene columns missing"
assert feature_column_alignment_audit["cell_columns_present"], "cell columns missing"
assert feature_column_alignment_audit["submission_target_columns_match_scored_targets"], "submission target columns do not match scored targets"

assert missing_value_audit["missing_values"].sum() == 0, "Missing values found in loaded clean data"

print("Step 2 hard checks passed successfully.")

Step 2 hard checks passed successfully.


In [19]:
data_shape_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_data_shape_audit.csv",
    index=False
)

feature_group_count_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_feature_group_count_audit.csv",
    index=False
)

pd.DataFrame([id_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_2_id_alignment_audit.csv",
    index=False
)

pd.DataFrame([feature_column_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_2_feature_column_alignment_audit.csv",
    index=False
)

missing_value_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_missing_value_audit.csv",
    index=False
)

print("Step 2 data loading and audit files saved successfully.")

Step 2 data loading and audit files saved successfully.


In [20]:
step_2_status = {
    "step": "Step 2 — Load Clean Interim Data",
    "status": "completed",
    "clean_data_loaded": True,
    "feature_groups_loaded": True,
    "id_alignment_checked": True,
    "feature_column_alignment_checked": True,
    "missing_values_checked": True,
    "hard_checks_passed": True,
    "ready_for_step_3": True,
    "next_step": "Step 3 — Feature Group Verification"
}

step_2_status

{'step': 'Step 2 — Load Clean Interim Data',
 'status': 'completed',
 'clean_data_loaded': True,
 'feature_groups_loaded': True,
 'id_alignment_checked': True,
 'feature_column_alignment_checked': True,
 'missing_values_checked': True,
 'hard_checks_passed': True,
 'ready_for_step_3': True,
 'next_step': 'Step 3 — Feature Group Verification'}

## Step 3 — Feature Group Verification and Forbidden Column Pre-Audit

This step verifies that all feature groups are correctly separated before feature engineering.

The goal is to make sure that:

- metadata, gene, and cell features are present;
- scored and nonscored targets are separated from input features;
- `drug_id` is not used as a baseline input feature;
- no forbidden or target-derived feature can accidentally enter the model input matrix.

In [21]:
feature_group_verification = pd.DataFrame({
    "group_name": [
        "ID column",
        "Metadata features",
        "Gene features",
        "Cell features",
        "Scored target features",
        "Nonscored target features"
    ],
    "expected_pattern": [
        "sig_id",
        "cp_type, cp_time, cp_dose",
        "columns starting with g-",
        "columns starting with c-",
        "columns from y_scored excluding sig_id",
        "columns from y_nonscored excluding sig_id"
    ],
    "count": [
        1,
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features)
    ],
    "status": [
        ID_COL in train_features_clean.columns,
        all(col in train_features_clean.columns for col in metadata_features),
        all(col in train_features_clean.columns for col in gene_features),
        all(col in train_features_clean.columns for col in cell_features),
        all(col in y_scored.columns for col in scored_target_features),
        all(col in y_nonscored.columns for col in nonscored_target_features)
    ]
})

feature_group_verification

,group_name,expected_pattern,count,status
0,ID column,sig_id,1,True
1,Metadata features,"cp_type, cp_time, cp_dose",3,True
2,Gene features,columns starting with g-,772,True
3,Cell features,columns starting with c-,100,True
4,Scored target features,columns from y_scored excluding sig_id,206,True
5,Nonscored target features,columns from y_nonscored excluding sig_id,402,True


In [22]:
gene_cell_naming_audit = pd.DataFrame({
    "check": [
        "All gene features start with g-",
        "All cell features start with c-",
        "No gene feature appears in cell features",
        "No cell feature appears in gene features",
        "Gene features exist in train",
        "Gene features exist in test",
        "Cell features exist in train",
        "Cell features exist in test"
    ],
    "passed": [
        all(col.startswith("g-") for col in gene_features),
        all(col.startswith("c-") for col in cell_features),
        len(set(gene_features).intersection(set(cell_features))) == 0,
        len(set(cell_features).intersection(set(gene_features))) == 0,
        all(col in train_features_clean.columns for col in gene_features),
        all(col in test_features_clean.columns for col in gene_features),
        all(col in train_features_clean.columns for col in cell_features),
        all(col in test_features_clean.columns for col in cell_features)
    ]
})

gene_cell_naming_audit

,check,passed
0,All gene features start with g-,True
1,All cell features start with c-,True
2,No gene feature appears in cell features,True
3,No cell feature appears in gene features,True
4,Gene features exist in train,True
5,Gene features exist in test,True
6,Cell features exist in train,True
7,Cell features exist in test,True


In [23]:
duplicate_column_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean"
    ],
    "duplicate_column_count": [
        train_features_clean.columns.duplicated().sum(),
        test_features_clean.columns.duplicated().sum(),
        y_scored.columns.duplicated().sum(),
        y_nonscored.columns.duplicated().sum(),
        train_drug_clean.columns.duplicated().sum()
    ]
})

duplicate_column_audit

,dataset,duplicate_column_count
0,train_features_clean,0
1,test_features_clean,0
2,y_scored,0
3,y_nonscored,0
4,train_drug_clean,0


In [24]:
forbidden_input_columns = set(scored_target_features) | set(nonscored_target_features) | {
    "drug_id",
    "active_target_count",
    "zero_label_flag",
    "multi_label_count",
    "scored_positive_count",
    "nonscored_positive_count",
    "target_frequency",
    "fold_positive_count",
    "validation_positive_count",
    "target_cooccurrence_count"
}

forbidden_input_columns_preview = pd.DataFrame({
    "forbidden_column_or_pattern": sorted(list(forbidden_input_columns))[:30]
})

forbidden_input_columns_preview

,forbidden_column_or_pattern
0,11-beta-hsd1_inhibitor
1,5-alpha_reductase_inhibitor
2,abc_transporter_expression_enhancer
3,abl_inhibitor
4,acat_inhibitor
5,ace_inhibitor
6,acetylcholine_receptor_agonist
7,acetylcholine_receptor_antagonist
8,acetylcholine_release_enhancer
9,acetylcholinesterase_inhibitor


In [25]:
train_forbidden_columns_found = sorted(
    list(set(train_features_clean.columns).intersection(forbidden_input_columns))
)

test_forbidden_columns_found = sorted(
    list(set(test_features_clean.columns).intersection(forbidden_input_columns))
)

forbidden_column_pre_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean"
    ],
    "forbidden_columns_found_count": [
        len(train_forbidden_columns_found),
        len(test_forbidden_columns_found)
    ],
    "forbidden_columns_found": [
        train_forbidden_columns_found,
        test_forbidden_columns_found
    ],
    "passed": [
        len(train_forbidden_columns_found) == 0,
        len(test_forbidden_columns_found) == 0
    ]
})

forbidden_column_pre_audit

,dataset,forbidden_columns_found_count,forbidden_columns_found,passed
0,train_features_clean,0,[],True
1,test_features_clean,0,[],True


In [26]:
target_separation_audit = pd.DataFrame({
    "check": [
        "No scored target appears in train_features_clean",
        "No scored target appears in test_features_clean",
        "No nonscored target appears in train_features_clean",
        "No nonscored target appears in test_features_clean",
        "Scored targets exist only in y_scored",
        "Nonscored targets exist only in y_nonscored"
    ],
    "passed": [
        len(set(scored_target_features).intersection(train_features_clean.columns)) == 0,
        len(set(scored_target_features).intersection(test_features_clean.columns)) == 0,
        len(set(nonscored_target_features).intersection(train_features_clean.columns)) == 0,
        len(set(nonscored_target_features).intersection(test_features_clean.columns)) == 0,
        all(col in y_scored.columns for col in scored_target_features),
        all(col in y_nonscored.columns for col in nonscored_target_features)
    ]
})

target_separation_audit

,check,passed
0,No scored target appears in train_features_clean,True
1,No scored target appears in test_features_clean,True
2,No nonscored target appears in train_features_...,True
3,No nonscored target appears in test_features_c...,True
4,Scored targets exist only in y_scored,True
5,Nonscored targets exist only in y_nonscored,True


In [27]:
drug_id_handling_audit = pd.DataFrame({
    "check": [
        "drug_id exists in train_drug_clean",
        "drug_id does not exist in train_features_clean",
        "drug_id does not exist in test_features_clean",
        "drug_id will not be used in baseline feature matrices",
        "drug_id is available only for validation diagnostics"
    ],
    "passed": [
        "drug_id" in train_drug_clean.columns,
        "drug_id" not in train_features_clean.columns,
        "drug_id" not in test_features_clean.columns,
        True,
        True
    ],
    "reason": [
        "Needed for drug-overlap and group-validation diagnostics",
        "Prevents accidental baseline input leakage",
        "Test-side standard feature matrix should not depend on drug_id",
        "Baseline X should use metadata + gene + cell + deterministic summaries only",
        "Drug-aware validation can be tested later as robustness analysis"
    ]
})

drug_id_handling_audit

,check,passed,reason
0,drug_id exists in train_drug_clean,True,Needed for drug-overlap and group-validation d...
1,drug_id does not exist in train_features_clean,True,Prevents accidental baseline input leakage
2,drug_id does not exist in test_features_clean,True,Test-side standard feature matrix should not d...
3,drug_id will not be used in baseline feature m...,True,Baseline X should use metadata + gene + cell +...
4,drug_id is available only for validation diagn...,True,Drug-aware validation can be tested later as r...


In [28]:
assert feature_group_verification["status"].all(), "Some feature groups are missing or invalid"
assert gene_cell_naming_audit["passed"].all(), "Gene/cell feature naming audit failed"
assert duplicate_column_audit["duplicate_column_count"].sum() == 0, "Duplicate columns found"
assert forbidden_column_pre_audit["passed"].all(), "Forbidden columns found in raw input features"
assert target_separation_audit["passed"].all(), "Target separation audit failed"
assert drug_id_handling_audit["passed"].all(), "Drug ID handling audit failed"

print("Step 3 feature group verification and forbidden-column pre-audit passed successfully.")

Step 3 feature group verification and forbidden-column pre-audit passed successfully.


In [29]:
feature_group_verification.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_feature_group_verification.csv",
    index=False
)

gene_cell_naming_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_gene_cell_naming_audit.csv",
    index=False
)

duplicate_column_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_duplicate_column_audit.csv",
    index=False
)

forbidden_column_pre_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_forbidden_column_pre_audit.csv",
    index=False
)

target_separation_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_target_separation_audit.csv",
    index=False
)

drug_id_handling_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_drug_id_handling_audit.csv",
    index=False
)

print("Step 3 audit files saved successfully.")

Step 3 audit files saved successfully.


In [30]:
step_3_status = {
    "step": "Step 3 — Feature Group Verification and Forbidden Column Pre-Audit",
    "status": "completed",
    "feature_groups_verified": True,
    "gene_cell_features_verified": True,
    "duplicate_columns_checked": True,
    "target_separation_checked": True,
    "forbidden_columns_checked": True,
    "drug_id_excluded_from_baseline": True,
    "ready_for_step_4": True,
    "next_step": "Step 4 — Build FE_V0 Raw Baseline Matrix"
}

step_3_status

{'step': 'Step 3 — Feature Group Verification and Forbidden Column Pre-Audit',
 'status': 'completed',
 'feature_groups_verified': True,
 'gene_cell_features_verified': True,
 'duplicate_columns_checked': True,
 'target_separation_checked': True,
 'forbidden_columns_checked': True,
 'drug_id_excluded_from_baseline': True,
 'ready_for_step_4': True,
 'next_step': 'Step 4 — Build FE_V0 Raw Baseline Matrix'}